In [1]:
import pandas as pd
import numpy as np
import random
import ast
import torch
from datetime import timedelta
from sklearn.metrics.pairwise import cosine_similarity
import pickle

### Only Similarity (no coverage)

In [ ]:
with open("../../datasets/.pkl/window_topics.pkl", "rb") as f:
    window_topics = pickle.load(f)

In [3]:
relations = []
similarity_threshold = 0.7
coverage_threshold_forward = 0.3
coverage_threshold_backward = 0.3

In [4]:
for i in range(len(window_topics)-1):
    w1, w2 = window_topics[i], window_topics[i+1]
    if not w1["topic_embeddings"] or not w2["topic_embeddings"]:
        continue

    t1_ids = [tid for tid in w1["topic_embeddings"].keys() if tid != -1]
    t2_ids = [tid for tid in w2["topic_embeddings"].keys() if tid != -1]
    
    emb1 = np.array([w1["topic_embeddings"][tid] for tid in t1_ids])
    emb2 = np.array([w2["topic_embeddings"][tid] for tid in t2_ids])

    sim_matrix = cosine_similarity(emb1, emb2)
    sim_matrix = np.clip(sim_matrix, a_min=0, a_max=None)

    for idx1, tid1 in enumerate(t1_ids):
        sims = sim_matrix[idx1]

        candidate_indices = [
            idx2 for idx2, sim in enumerate(sims)
            if sim >= similarity_threshold
        ]

        if not candidate_indices:
            relations.append({
                "window1": i,
                "topic1": tid1,
                "topic1_words": w1["topic_words"].get(tid1, []),
                "window2": i+1,
                "successors": [],
                "successors_words": [],
                "successors_similarities": [],
                "coverage_forward": [],
                "coverage_backward": [],
                "coverage_sum_forward": 0,
                "coverage_sum_backward": 0,
                "relation": "disappeared"
            })
            continue

        sim_subset = sims[candidate_indices]
        denom_out = sim_subset.sum()

        connected = []
        cover_f_vals = []
        cover_b_vals = []

        for idx2 in candidate_indices:
            cov_f = sim_matrix[idx1, idx2] / denom_out if denom_out > 0 else 0

            col_sims = sim_matrix[:, idx2]
            col_candidates = [
                idx1b for idx1b, sim in enumerate(col_sims)
                if sim >= similarity_threshold
            ]
            denom_in = col_sims[col_candidates].sum()

            cov_b = sim_matrix[idx1, idx2] / denom_in if denom_in > 0 else 0

            if cov_f >= coverage_threshold_forward and cov_b >= coverage_threshold_backward:
                connected.append(t2_ids[idx2])
                cover_f_vals.append(cov_f)
                cover_b_vals.append(cov_b)

        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if not connected or total_f < 0.4 or total_b < 0.4:
            relation = "disappeared"
        elif len(connected)==1 and cover_f_vals[0]>0.5 and cover_b_vals[0]>0.5 and sims[t2_ids.index(connected[0])]>0.9:
            relation = "continued"
        elif len(connected)>=2 and total_f>=0.7 and all(cv>0.5 for cv in cover_b_vals):
            relation = "split"
        else:
            relation = "unclear"

        relations.append({
            "window1": i,
            "topic1": tid1,
            "topic1_words": w1["topic_words"].get(tid1, []),
            "window2": i+1,
            "successors": connected,
            "successors_words": [w2["topic_words"].get(t2, []) for t2 in connected],
            "successors_similarities": [round(sims[t2_ids.index(t2)],3) for t2 in connected],
            "coverage_forward": [round(cv,3) for cv in cover_f_vals],
            "coverage_backward": [round(cv,3) for cv in cover_b_vals],
            "coverage_sum_forward": round(total_f,3),
            "coverage_sum_backward": round(total_b,3),
            "relation": relation
        })

    for idx2, tid2 in enumerate(t2_ids):
        sims_col = sim_matrix[:, idx2]

        candidate_indices = [
            idx1 for idx1, sim in enumerate(sims_col)
            if sim >= similarity_threshold
        ]

        if len(candidate_indices) < 2:
            continue

        sim_subset = sims_col[candidate_indices]
        denom_in = sim_subset.sum()

        connected2 = []
        cover_f_vals = []
        cover_b_vals = []

        for idx1 in candidate_indices:
            cov_b = sim_matrix[idx1, idx2] / denom_in if denom_in > 0 else 0

            row_sims = sim_matrix[idx1]
            row_candidates = [
                idx2b for idx2b, sim in enumerate(row_sims)
                if sim >= similarity_threshold
            ]
            denom_out = row_sims[row_candidates].sum()

            cov_f = sim_matrix[idx1, idx2] / denom_out if denom_out > 0 else 0

            if cov_f >= coverage_threshold_forward and cov_b >= coverage_threshold_backward:
                connected2.append(t1_ids[idx1])
                cover_f_vals.append(cov_f)
                cover_b_vals.append(cov_b)

        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if len(connected2)>=2 and total_b>=0.7 and all(cv>0.5 for cv in cover_f_vals):
            relations.append({
                "window1": i,
                "topics_from": connected2,
                "topics_from_words": [w1["topic_words"].get(tid,[]) for tid in connected2],
                "window2": i+1,
                "topic_to": tid2,
                "topic_to_words": w2["topic_words"].get(tid2,[]),
                "antecedents_similarities": [round(sim_matrix[t1_ids.index(tid), idx2],3) for tid in connected2],
                "coverage_forward": [round(cv,3) for cv in cover_f_vals],
                "coverage_backward": [round(cv,3) for cv in cover_b_vals],
                "coverage_sum_forward": round(total_f,3),
                "coverage_sum_backward": round(total_b,3),
                "relation": "merge"
            })

merged_from_pairs = set()
for r in relations:
    if r.get("relation")=="merge":
        merged_from_pairs.update([(r["window1"], tid) for tid in r["topics_from"]])

relations = [
    r for r in relations
    if not (
        r["relation"] in ["disappeared","unclear","continued"]
        and (r["window1"], r.get("topic1",-1)) in merged_from_pairs
    )
]

df_final_relations = pd.DataFrame(relations)

In [5]:
df_final_relations['relation'].value_counts()

relation
disappeared    934
continued      853
unclear        376
merge           90
split           90
Name: count, dtype: int64

In [ ]:
df_final_relations.to_csv("../../datasets/bertilda_mergeandsplits_nocoverage.csv", index=False)

### Lexical-only

In [ ]:
with open("../../datasets/.pkl/window_topics.pkl", "rb") as f:
    window_topics = pickle.load(f)

In [8]:
relations = []
similarity_threshold = 0.7
coverage_threshold_forward = 0.3
coverage_threshold_backward = 0.3

In [9]:
def jaccard_similarity(words1, words2):
    set1, set2 = set(words1), set(words2)
    union = set1 | set2
    if not union:
        return 0.0
    return len(set1 & set2) / len(union)

In [10]:
for i in range(len(window_topics)-1):
    w1, w2 = window_topics[i], window_topics[i+1]
    if not w1["topic_words"] or not w2["topic_words"]:
        continue

    t1_ids = [tid for tid in w1["topic_words"].keys() if tid != -1]
    t2_ids = [tid for tid in w2["topic_words"].keys() if tid != -1]

    sim_matrix = np.zeros((len(t1_ids), len(t2_ids)))

    for idx1, tid1 in enumerate(t1_ids):
        words1 = w1["topic_words"][tid1]
        for idx2, tid2 in enumerate(t2_ids):
            words2 = w2["topic_words"][tid2]
            sim_matrix[idx1, idx2] = jaccard_similarity(words1, words2)

    sim_matrix = np.clip(sim_matrix, a_min=0, a_max=None)

    for idx1, tid1 in enumerate(t1_ids):
        sims = sim_matrix[idx1]

        candidate_indices = [
            idx2 for idx2, sim in enumerate(sims)
            if sim >= similarity_threshold
        ]

        if not candidate_indices:
            relations.append({
                "window1": i,
                "topic1": tid1,
                "topic1_words": w1["topic_words"].get(tid1, []),
                "window2": i+1,
                "successors": [],
                "successors_words": [],
                "successors_similarities": [],
                "coverage_forward": [],
                "coverage_backward": [],
                "coverage_sum_forward": 0,
                "coverage_sum_backward": 0,
                "relation": "disappeared"
            })
            continue

        sim_subset = sims[candidate_indices]
        denom_out = sim_subset.sum()

        connected = []
        cover_f_vals = []
        cover_b_vals = []

        for idx2 in candidate_indices:
            cov_f = sim_matrix[idx1, idx2] / denom_out if denom_out > 0 else 0

            col_sims = sim_matrix[:, idx2]
            col_candidates = [
                idx1b for idx1b, sim in enumerate(col_sims)
                if sim >= similarity_threshold
            ]
            denom_in = col_sims[col_candidates].sum()

            cov_b = sim_matrix[idx1, idx2] / denom_in if denom_in > 0 else 0

            if cov_f >= coverage_threshold_forward and cov_b >= coverage_threshold_backward:
                connected.append(t2_ids[idx2])
                cover_f_vals.append(cov_f)
                cover_b_vals.append(cov_b)

        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if not connected or total_f < 0.4 or total_b < 0.4:
            relation = "disappeared"
        elif len(connected)==1 and cover_f_vals[0]>0.5 and cover_b_vals[0]>0.5 and sims[t2_ids.index(connected[0])]>0.9:
            relation = "continued"
        elif len(connected)>=2 and total_f>=0.7 and all(cv>0.5 for cv in cover_b_vals):
            relation = "split"
        else:
            relation = "unclear"

        relations.append({
            "window1": i,
            "topic1": tid1,
            "topic1_words": w1["topic_words"].get(tid1, []),
            "window2": i+1,
            "successors": connected,
            "successors_words": [w2["topic_words"].get(t2, []) for t2 in connected],
            "successors_similarities": [round(sims[t2_ids.index(t2)],3) for t2 in connected],
            "coverage_forward": [round(cv,3) for cv in cover_f_vals],
            "coverage_backward": [round(cv,3) for cv in cover_b_vals],
            "coverage_sum_forward": round(total_f,3),
            "coverage_sum_backward": round(total_b,3),
            "relation": relation
        })

    for idx2, tid2 in enumerate(t2_ids):
        sims_col = sim_matrix[:, idx2]

        candidate_indices = [
            idx1 for idx1, sim in enumerate(sims_col)
            if sim >= similarity_threshold
        ]

        if len(candidate_indices) < 2:
            continue

        sim_subset = sims_col[candidate_indices]
        denom_in = sim_subset.sum()

        connected2 = []
        cover_f_vals = []
        cover_b_vals = []

        for idx1 in candidate_indices:
            cov_b = sim_matrix[idx1, idx2] / denom_in if denom_in > 0 else 0

            row_sims = sim_matrix[idx1]
            row_candidates = [
                idx2b for idx2b, sim in enumerate(row_sims)
                if sim >= similarity_threshold
            ]
            denom_out = row_sims[row_candidates].sum()

            cov_f = sim_matrix[idx1, idx2] / denom_out if denom_out > 0 else 0

            if cov_f >= coverage_threshold_forward and cov_b >= coverage_threshold_backward:
                connected2.append(t1_ids[idx1])
                cover_f_vals.append(cov_f)
                cover_b_vals.append(cov_b)

        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if len(connected2)>=2 and total_b>=0.7 and all(cv>0.5 for cv in cover_f_vals):
            relations.append({
                "window1": i,
                "topics_from": connected2,
                "topics_from_words": [w1["topic_words"].get(tid,[]) for tid in connected2],
                "window2": i+1,
                "topic_to": tid2,
                "topic_to_words": w2["topic_words"].get(tid2,[]),
                "antecedents_similarities": [round(sim_matrix[t1_ids.index(tid), idx2],3) for tid in connected2],
                "coverage_forward": [round(cv,3) for cv in cover_f_vals],
                "coverage_backward": [round(cv,3) for cv in cover_b_vals],
                "coverage_sum_forward": round(total_f,3),
                "coverage_sum_backward": round(total_b,3),
                "relation": "merge"
            })

merged_from_pairs = set()
for r in relations:
    if r.get("relation")=="merge":
        merged_from_pairs.update([(r["window1"], tid) for tid in r["topics_from"]])

relations = [
    r for r in relations
    if not (
        r["relation"] in ["disappeared","unclear","continued"]
        and (r["window1"], r.get("topic1",-1)) in merged_from_pairs
    )
]

df_final_relations = pd.DataFrame(relations)

In [11]:
df_final_relations['relation'].value_counts()

relation
disappeared    1906
unclear         334
continued       198
Name: count, dtype: int64

In [ ]:
df_final_relations.to_csv("../../datasets/bertilda_mergeandsplits_lexicalonly.csv", index=False)

### Foward-only

In [ ]:
with open("../../datasets/.pkl/window_topics.pkl", "rb") as f:
    window_topics = pickle.load(f)

In [14]:
relations = []
tweet_to_centroid_threshold = 0.4
similarity_threshold = 0.7
coverage_threshold_forward = 0.3
coverage_threshold_backward = 0.3

In [15]:
def compute_forward_flows(w1, w2):
    flows = {}

    w1_docs = pd.DataFrame(w1["tweets_with_topics"])
    w1_docs = w1_docs[w1_docs["topic_id"] != -1]

    t2_ids = [tid for tid in w2["topic_embeddings"].keys() if tid != -1]
    centroids_new = np.array([w2["topic_embeddings"][tid] for tid in t2_ids])

    for tid1 in [tid for tid in w1["topic_embeddings"].keys() if tid != -1]:

        docs_t1 = w1_docs[w1_docs["topic_id"] == tid1]
        if docs_t1.empty:
            continue

        tweet_embs = np.vstack(docs_t1["embedding"].values)
        sim_matrix = cosine_similarity(tweet_embs, centroids_new)

        best_matches = sim_matrix.argmax(axis=1)
        best_scores = sim_matrix.max(axis=1)

        valid_idx = best_scores >= tweet_to_centroid_threshold
        best_matches = best_matches[valid_idx]

        counts_by_idx = np.bincount(best_matches, minlength=len(t2_ids))

        for idx2, cnt in enumerate(counts_by_idx):
            if cnt > 0:
                flows[(tid1, t2_ids[idx2])] = cnt

    return flows

In [16]:
# ============================================================================================================================================================
# Main Loop w1 → w2
# ============================================================================================================================================================

for i in range(len(window_topics)-1):
    w1, w2 = window_topics[i], window_topics[i+1]
    if not w1["topic_embeddings"] or not w2["topic_embeddings"]:
        continue

    t1_ids = [tid for tid in w1["topic_words"].keys() if tid != -1]
    t2_ids = [tid for tid in w2["topic_words"].keys() if tid != -1]
    emb1 = np.array([w1["topic_embeddings"][tid] for tid in t1_ids])
    emb2 = np.array([w2["topic_embeddings"][tid] for tid in t2_ids])
    sim_matrix = cosine_similarity(emb1, emb2)

    flows = compute_forward_flows(w1, w2)

    coverage_f = {}
    coverage_b = {}

    for (tid1, tid2), F in flows.items():

        n_old = w1["topic_doc_counts"].get(tid1, 1)
        n_new = w2["topic_doc_counts"].get(tid2, 1)

        coverage_f[(tid1, tid2)] = F / n_old
        coverage_b[(tid1, tid2)] = F / n_new

    # Relations: continued, split, disappeared
    for idx1, tid1 in enumerate(t1_ids):
        sims = sim_matrix[idx1]
        connected = [
            t2_ids[idx2]
            for idx2, sim in enumerate(sims)
            if sim >= similarity_threshold
            and coverage_f.get((tid1, t2_ids[idx2]), 0) >= coverage_threshold_forward
            and coverage_b.get((tid1, t2_ids[idx2]), 0) >= coverage_threshold_backward
        ]
        cover_f_vals = [coverage_f.get((tid1,t2),0) for t2 in connected]
        cover_b_vals = [coverage_b.get((tid1,t2),0) for t2 in connected]
        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if not connected or total_f < 0.4 or total_b < 0.4:
            relation = "disappeared"
        elif len(connected)==1 and cover_f_vals[0]>0.5 and cover_b_vals[0]>0.5 and sims[t2_ids.index(connected[0])]>0.9:
            relation = "continued"
        elif len(connected)>=2 and total_f>=0.7 and all(cv>0.5 for cv in cover_b_vals):
            relation = "split"
        else:
            relation = "unclear"

        relations.append({
            "window1": i,
            "topic1": tid1,
            "topic1_words": w1["topic_words"].get(tid1, []),
            "window2": i+1,
            "successors": connected,
            "successors_words": [w2["topic_words"].get(t2, []) for t2 in connected],
            "successors_similarities": [round(sims[t2_ids.index(t2)],3) for t2 in connected],
            "coverage_forward": [round(cv,3) for cv in cover_f_vals],
            "coverage_backward": [round(cv,3) for cv in cover_b_vals],
            "coverage_sum_forward": round(total_f,3),
            "coverage_sum_backward": round(total_b,3),
            "relation": relation
        })

    # Merge detection
    for idx2, tid2 in enumerate(t2_ids):
        sims = sim_matrix[:, idx2]
        connected2 = [
            t1_ids[idx1]
            for idx1, sim in enumerate(sims)
            if sim >= similarity_threshold
            and coverage_f.get((t1_ids[idx1], tid2), 0) >= coverage_threshold_forward
            and coverage_b.get((t1_ids[idx1], tid2), 0) >= coverage_threshold_backward
        ]
        cover_f_vals = [coverage_f.get((t1,t2_ids[idx2]),0) for t1 in connected2]
        cover_b_vals = [coverage_b.get((t1,t2_ids[idx2]),0) for t1 in connected2]
        total_f = sum(cover_f_vals)
        total_b = sum(cover_b_vals)

        if len(connected2)>=2 and total_b>=0.7 and all(cv>0.5 for cv in cover_f_vals):
            relations.append({
                "window1": i,
                "topics_from": connected2,
                "topics_from_words": [w1["topic_words"].get(tid,[]) for tid in connected2],
                "window2": i+1,
                "topic_to": tid2,
                "topic_to_words": w2["topic_words"].get(tid2,[]),
                "antecedents_similarities": [round(sim_matrix[t1_ids.index(tid), idx2],3) for tid in connected2],
                "coverage_forward": [round(cv,3) for cv in cover_f_vals],
                "coverage_backward": [round(cv,3) for cv in cover_b_vals],
                "coverage_sum_forward": round(total_f,3),
                "coverage_sum_backward": round(total_b,3),
                "relation": "merge"
            })

# Remove false positives
merged_from_pairs = set()
for r in relations:
    if r.get("relation")=="merge":
        merged_from_pairs.update([(r["window1"], tid) for tid in r["topics_from"]])

relations = [
    r for r in relations
    if not (
        r["relation"] in ["disappeared","unclear", "continued"] and (r["window1"], r.get("topic1",-1)) in merged_from_pairs
    )
]

df_final_relations = pd.DataFrame(relations)

In [18]:
df_final_relations['relation'].value_counts()

relation
continued      1291
disappeared     893
unclear         153
merge            35
split            29
Name: count, dtype: int64

In [ ]:
df_final_relations.to_csv("../../datasets/bertilda_mergeandsplits_Forward-only.csv", index=False)